# Google Play Controlled Scale Ingestion — Phase 2 Day 1

This notebook starts Phase 2 of the Google Play review ingestion pipeline.

The earlier work already showed that the basic ingestion and database flow can run. In this notebook, I am testing a larger controlled batch to see whether the same database can keep growing in a clean and reliable way.

Main checks in this run:

- more apps, not only the original small app set
- larger review volume per app
- continued ingestion from the existing SQLite database
- duplicate handling using `source + app_id + review_id`
- raw review table, cleaned review table, ingestion run tracking, and quality flag linkage
- runtime, inserted rows, duplicate rows, errors, quality flags, and database growth
- simple run summary after the ingestion run

Existing database used in this notebook:

`database/google_play_reviews.sqlite`

The goal is not to rebuild the whole project from zero. The goal is to check whether the current pipeline can move from a small controlled demo into a more realistic recurring ingestion process.

## 1. Setup

I first clone the current GitHub repo and work inside the repo folder.

If the repo is already cloned in Colab, this cell will not clone it again.

In [1]:
import os

REPO_URL = "https://github.com/Yaxuanzhang5/app-review-source-validation.git"
REPO_DIR = "app-review-source-validation"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print("Repo folder already exists.")

%cd {REPO_DIR}

Cloning into 'app-review-source-validation'...
remote: Enumerating objects: 272, done.
remote: Total 272 (delta 0), reused 0 (delta 0), pack-reused 272 (from 1)
Receiving objects: 100% (272/272), 2.63 MiB | 5.19 MiB/s, done.
Resolving deltas: 100% (123/123), done.
/content/app-review-source-validation


## 2. Install packages

For this notebook, I keep the package setup simple.

Main package:

- `google-play-scraper`: fetches Google Play app metadata and reviews

Other packages are standard Python packages for SQLite, timestamps, summaries, and saving output files.

In [2]:
!pip install -q google-play-scraper pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.6 MB/s eta 0:00:00


In [3]:
import os
import re
import json
import time
import html
import shutil
import sqlite3
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from google_play_scraper import app, reviews, Sort

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

print("Packages imported.")

Packages imported.


## 3. Run configuration

This is the first controlled scale run for Phase 2.

I use 10 apps and 1,200 target reviews per app. This is larger than the earlier small demo, but still controlled enough to review safely.

The `FREQUENCY_LABEL` is important because later I can run the same notebook again with a different label, such as `twice_daily_pm` or `daily_followup`.

In [4]:
PHASE = "phase2"
RUN_LABEL = "phase2_day1_controlled_scale"
FREQUENCY_LABEL = "once_daily_baseline"

SOURCE = "google_play"
LANGUAGE = "en"
COUNTRY = "us"
TARGET_REVIEWS_PER_APP = 1200

DB_PATH = "database/google_play_reviews.sqlite"

OUTPUT_DIR = Path("outputs")
RUN_SUMMARY_DIR = OUTPUT_DIR / "run_summaries"
QUALITY_DIR = OUTPUT_DIR / "quality"
REPORT_DIR = Path("reports")
BACKUP_DIR = Path("database") / "backups"

for folder in [RUN_SUMMARY_DIR, QUALITY_DIR, REPORT_DIR, BACKUP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

APPS = {
    "YouTube": "com.google.android.youtube",
    "TikTok": "com.zhiliaoapp.musically",
    "Spotify": "com.spotify.music",
    "Instagram": "com.instagram.android",
    "Uber": "com.ubercab",
    "DoorDash": "com.dd.doordash",
    "Duolingo": "com.duolingo",
    "Google Maps": "com.google.android.apps.maps",
    "Netflix": "com.netflix.mediaclient",
    "Reddit": "com.reddit.frontpage"
}

run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_ID = f"{RUN_LABEL}_{run_timestamp}"

print("Run ID:", RUN_ID)
print("Database path:", DB_PATH)
print("App count:", len(APPS))
print("Target reviews per app:", TARGET_REVIEWS_PER_APP)

Run ID: phase2_day1_controlled_scale_20260708_034445
Database path: database/google_play_reviews.sqlite
App count: 10
Target reviews per app: 1200


## 4. Confirm the existing database and create a backup

I do not want to start from an empty database here.

The point of this phase is to continue from the existing database and check whether it grows in a reasonable way.

Before making any database changes, I save one backup copy.

In [5]:
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f"Database file not found: {DB_PATH}")

def get_db_size_mb(path):
    if os.path.exists(path):
        return os.path.getsize(path) / (1024 * 1024)
    return 0

db_size_before_backup_mb = get_db_size_mb(DB_PATH)
backup_path = BACKUP_DIR / f"google_play_reviews_before_{RUN_ID}.sqlite"

shutil.copy2(DB_PATH, backup_path)

print(f"Database exists: {DB_PATH}")
print(f"Current database size before backup: {db_size_before_backup_mb:.4f} MB")
print(f"Backup saved to: {backup_path}")

Database exists: database/google_play_reviews.sqlite
Current database size before backup: 1.8320 MB
Backup saved to: database/backups/google_play_reviews_before_phase2_day1_controlled_scale_20260708_034445.sqlite


## 5. Inspect the current database before the new run

Before adding Phase 2 data, I check the current tables, row counts, schemas, and indexes.

This step is important because I want to confirm that this notebook is using the existing database instead of silently creating a new one.

In [6]:
def get_tables(conn):
    query = """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """
    return pd.read_sql_query(query, conn)

def table_exists(conn, table_name):
    query = """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table' AND name = ?;
    """
    result = pd.read_sql_query(query, conn, params=[table_name])
    return len(result) > 0

def count_rows(conn, table_name):
    if not table_exists(conn, table_name):
        return 0

    query = f"SELECT COUNT(*) AS row_count FROM {table_name};"
    return int(pd.read_sql_query(query, conn)["row_count"].iloc[0])

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

tables_before_df = get_tables(conn)

print("Tables before Phase 2 Day 1:")
display(tables_before_df)

row_counts_before = []

for table_name in tables_before_df["name"]:
    row_counts_before.append({
        "table_name": table_name,
        "row_count_before": count_rows(conn, table_name)
    })

row_counts_before_df = pd.DataFrame(row_counts_before)

print("Row counts before Phase 2 Day 1:")
display(row_counts_before_df)

Tables before Phase 2 Day 1:


,name
0,app_sources
1,ingestion_run_targets
2,ingestion_runs
3,phase2_app_run_summary
4,phase2_apps
5,phase2_ingestion_runs
6,phase2_quality_flags
7,phase2_reviews_cleaned
8,phase2_reviews_raw
9,review_quality_flags


Row counts before Phase 2 Day 1:


,table_name,row_count_before
0,app_sources,3
1,ingestion_run_targets,12
2,ingestion_runs,4
3,phase2_app_run_summary,10
4,phase2_apps,10
5,phase2_ingestion_runs,1
6,phase2_quality_flags,0
7,phase2_reviews_cleaned,0
8,phase2_reviews_raw,0
9,review_quality_flags,1200


In [7]:
for table_name in tables_before_df["name"]:
    print("\n" + "=" * 90)
    print(f"Schema for table: {table_name}")
    print("=" * 90)
    display(pd.read_sql_query(f"PRAGMA table_info({table_name});", conn))

    print(f"Indexes for table: {table_name}")
    indexes_df = pd.read_sql_query(f"PRAGMA index_list({table_name});", conn)
    display(indexes_df)

    if len(indexes_df) > 0:
        for index_name in indexes_df["name"]:
            print(f"Index details: {index_name}")
            display(pd.read_sql_query(f"PRAGMA index_info({index_name});", conn))


Schema for table: app_sources


,cid,name,type,notnull,dflt_value,pk
0,0,app_source_id,INTEGER,0,None,1
1,1,source_platform,TEXT,1,None,0
2,2,app_id,TEXT,1,None,0
3,3,app_name,TEXT,1,None,0
4,4,country,TEXT,1,None,0
5,5,language,TEXT,1,None,0
6,6,created_at,TEXT,1,None,0


Indexes for table: app_sources


,seq,name,unique,origin,partial
0,0,sqlite_autoindex_app_sources_1,1,u,0


Index details: sqlite_autoindex_app_sources_1


...

,seqno,cid,name
0,0,0,review_key



Schema for table: sqlite_sequence


,cid,name,type,notnull,dflt_value,pk
0,0,name,,0,None,0
1,1,seq,,0,None,0


Indexes for table: sqlite_sequence


,seq,name,unique,origin,partial


## 6. Create Phase 2 operational tables if needed

I keep the existing database file and add Phase 2 tables only if they do not already exist.

This avoids deleting or overwriting the earlier work, but still gives this phase a clean structure for:

- ingestion runs
- apps
- raw reviews
- cleaned reviews
- quality flags
- app-level summaries

The key duplicate rule is:

`source + app_id + review_id`

This matters because the same review should not be inserted again in later runs.

In [8]:
create_schema_sql = """
CREATE TABLE IF NOT EXISTS phase2_ingestion_runs (
    run_id TEXT PRIMARY KEY,
    run_label TEXT,
    phase TEXT,
    frequency_label TEXT,
    source TEXT,
    language TEXT,
    country TEXT,
    target_reviews_per_app INTEGER,
    app_count INTEGER,
    apps_included TEXT,
    run_started_at TEXT,
    run_finished_at TEXT,
    runtime_seconds REAL,
    status TEXT,
    records_fetched_total INTEGER DEFAULT 0,
    new_records_inserted_total INTEGER DEFAULT 0,
    duplicates_skipped_total INTEGER DEFAULT 0,
    errors_total INTEGER DEFAULT 0,
    apps_failed TEXT,
    quality_flag_total INTEGER DEFAULT 0,
    quality_flags_inserted INTEGER DEFAULT 0,
    db_size_before_mb REAL,
    db_size_after_mb REAL,
    db_size_growth_mb REAL,
    review_rows_before INTEGER,
    review_rows_after INTEGER,
    review_rows_growth INTEGER,
    notes TEXT
);

CREATE TABLE IF NOT EXISTS phase2_apps (
    app_id TEXT PRIMARY KEY,
    app_name TEXT,
    source TEXT,
    language TEXT,
    country TEXT,
    title_from_store TEXT,
    score_from_store REAL,
    ratings_from_store INTEGER,
    installs_from_store TEXT,
    last_validated_at TEXT,
    last_validation_status TEXT,
    last_validation_error TEXT
);

CREATE TABLE IF NOT EXISTS phase2_reviews_raw (
    review_key TEXT PRIMARY KEY,
    source TEXT NOT NULL,
    app_id TEXT NOT NULL,
    app_name TEXT,
    review_id TEXT,
    user_name TEXT,
    user_image TEXT,
    content_raw TEXT,
    score INTEGER,
    thumbs_up_count INTEGER,
    review_created_at TEXT,
    reply_content_raw TEXT,
    replied_at TEXT,
    app_version TEXT,
    fetched_at TEXT,
    run_id TEXT,
    raw_json TEXT,
    UNIQUE(source, app_id, review_id),
    FOREIGN KEY(app_id) REFERENCES phase2_apps(app_id),
    FOREIGN KEY(run_id) REFERENCES phase2_ingestion_runs(run_id)
);

CREATE TABLE IF NOT EXISTS phase2_reviews_cleaned (
    review_key TEXT PRIMARY KEY,
    source TEXT NOT NULL,
    app_id TEXT NOT NULL,
    content_cleaned TEXT,
    content_length INTEGER,
    has_developer_reply INTEGER,
    score INTEGER,
    review_created_at TEXT,
    app_version TEXT,
    cleaned_at TEXT,
    run_id TEXT,
    FOREIGN KEY(review_key) REFERENCES phase2_reviews_raw(review_key),
    FOREIGN KEY(run_id) REFERENCES phase2_ingestion_runs(run_id)
);

CREATE TABLE IF NOT EXISTS phase2_quality_flags (
    flag_id TEXT PRIMARY KEY,
    review_key TEXT,
    run_id TEXT,
    app_id TEXT,
    flag_name TEXT,
    flag_severity TEXT,
    flag_value TEXT,
    created_at TEXT,
    FOREIGN KEY(review_key) REFERENCES phase2_reviews_raw(review_key),
    FOREIGN KEY(run_id) REFERENCES phase2_ingestion_runs(run_id)
);

CREATE TABLE IF NOT EXISTS phase2_app_run_summary (
    run_id TEXT,
    app_name TEXT,
    app_id TEXT,
    target_reviews INTEGER,
    records_fetched INTEGER,
    unique_reviews_in_batch INTEGER,
    duplicate_reviews_in_batch INTEGER,
    new_records_inserted INTEGER,
    duplicates_skipped INTEGER,
    runtime_seconds REAL,
    min_review_date TEXT,
    max_review_date TEXT,
    missing_review_id_count INTEGER,
    missing_content_count INTEGER,
    empty_content_count INTEGER,
    missing_score_count INTEGER,
    invalid_score_count INTEGER,
    missing_review_date_count INTEGER,
    missing_app_version_count INTEGER,
    missing_developer_reply_count INTEGER,
    quality_flag_count INTEGER,
    error_message TEXT,
    PRIMARY KEY(run_id, app_id),
    FOREIGN KEY(run_id) REFERENCES phase2_ingestion_runs(run_id)
);
"""

conn.executescript(create_schema_sql)
conn.commit()

print("Phase 2 tables are ready.")
display(get_tables(conn))

Phase 2 tables are ready.


,name
0,app_sources
1,ingestion_run_targets
2,ingestion_runs
3,phase2_app_run_summary
4,phase2_apps
5,phase2_ingestion_runs
6,phase2_quality_flags
7,phase2_reviews_cleaned
8,phase2_reviews_raw
9,review_quality_flags


## 7. Start the ingestion run record

This inserts one row into the run tracking table before scraping starts.

At the end of the notebook, this same row will be updated with the final runtime, inserted rows, duplicate count, errors, quality flags, and database growth.

In [9]:
run_started_at = datetime.now(timezone.utc)
run_started_at_text = run_started_at.isoformat()

review_rows_before = count_rows(conn, "phase2_reviews_raw")
db_size_before_mb = get_db_size_mb(DB_PATH)

conn.execute(
    """
    INSERT OR REPLACE INTO phase2_ingestion_runs (
        run_id,
        run_label,
        phase,
        frequency_label,
        source,
        language,
        country,
        target_reviews_per_app,
        app_count,
        apps_included,
        run_started_at,
        status,
        db_size_before_mb,
        review_rows_before,
        notes
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """,
    (
        RUN_ID,
        RUN_LABEL,
        PHASE,
        FREQUENCY_LABEL,
        SOURCE,
        LANGUAGE,
        COUNTRY,
        TARGET_REVIEWS_PER_APP,
        len(APPS),
        ", ".join(APPS.keys()),
        run_started_at_text,
        "started",
        db_size_before_mb,
        review_rows_before,
        "Phase 2 Day 1 controlled scale ingestion run."
    )
)

conn.commit()

print("Run tracking row created.")
print("Run started at:", run_started_at_text)
print(f"Database size before run: {db_size_before_mb:.4f} MB")
print("Phase 2 raw review rows before run:", review_rows_before)

Run tracking row created.
Run started at: 2026-07-08T03:44:46.106977+00:00
Database size before run: 1.8320 MB
Phase 2 raw review rows before run: 0


## 8. Validate app IDs before scraping reviews

This step checks whether each app ID can return basic Google Play store metadata.

If one app has an issue, I record it instead of manually hiding it. This is useful because unstable apps or repeated failures are part of the operational test.

In [10]:
validation_rows = []
validated_at = datetime.now(timezone.utc).isoformat()

for app_name, app_id in APPS.items():
    try:
        store_info = app(
            app_id,
            lang=LANGUAGE,
            country=COUNTRY
        )

        validation_rows.append({
            "app_name": app_name,
            "app_id": app_id,
            "validation_status": "ok",
            "title_from_store": store_info.get("title"),
            "score_from_store": store_info.get("score"),
            "ratings_from_store": store_info.get("ratings"),
            "installs_from_store": store_info.get("installs"),
            "validation_error": ""
        })

        conn.execute(
            """
            INSERT INTO phase2_apps (
                app_id,
                app_name,
                source,
                language,
                country,
                title_from_store,
                score_from_store,
                ratings_from_store,
                installs_from_store,
                last_validated_at,
                last_validation_status,
                last_validation_error
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(app_id) DO UPDATE SET
                app_name = excluded.app_name,
                source = excluded.source,
                language = excluded.language,
                country = excluded.country,
                title_from_store = excluded.title_from_store,
                score_from_store = excluded.score_from_store,
                ratings_from_store = excluded.ratings_from_store,
                installs_from_store = excluded.installs_from_store,
                last_validated_at = excluded.last_validated_at,
                last_validation_status = excluded.last_validation_status,
                last_validation_error = excluded.last_validation_error;
            """,
            (
                app_id,
                app_name,
                SOURCE,
                LANGUAGE,
                COUNTRY,
                store_info.get("title"),
                store_info.get("score"),
                store_info.get("ratings"),
                store_info.get("installs"),
                validated_at,
                "ok",
                ""
            )
        )

    except Exception as e:
        error_text = str(e)

        validation_rows.append({
            "app_name": app_name,
            "app_id": app_id,
            "validation_status": "error",
            "title_from_store": None,
            "score_from_store": None,
            "ratings_from_store": None,
            "installs_from_store": None,
            "validation_error": error_text
        })

        conn.execute(
            """
            INSERT INTO phase2_apps (
                app_id,
                app_name,
                source,
                language,
                country,
                last_validated_at,
                last_validation_status,
                last_validation_error
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(app_id) DO UPDATE SET
                app_name = excluded.app_name,
                source = excluded.source,
                language = excluded.language,
                country = excluded.country,
                last_validated_at = excluded.last_validated_at,
                last_validation_status = excluded.last_validation_status,
                last_validation_error = excluded.last_validation_error;
            """,
            (
                app_id,
                app_name,
                SOURCE,
                LANGUAGE,
                COUNTRY,
                validated_at,
                "error",
                error_text
            )
        )

conn.commit()

validation_df = pd.DataFrame(validation_rows)

print("App validation result:")
display(validation_df)

App validation result:


,app_name,app_id,validation_status,title_from_store,score_from_store,ratings_from_store,installs_from_store,validation_error
0,YouTube,com.google.android.youtube,ok,YouTube,3.861219,170925967,"10,000,000,000+",
1,TikTok,com.zhiliaoapp.musically,ok,"TikTok - Videos, Shop & LIVE",3.991094,69280489,"1,000,000,000+",
2,Spotify,com.spotify.music,ok,Spotify: Music and Podcasts,4.335782,35890314,"1,000,000,000+",
3,Instagram,com.instagram.android,ok,Instagram,4.002019,168328553,"5,000,000,000+",
4,Uber,com.ubercab,ok,Uber - Request a ride,4.743543,19073517,"1,000,000,000+",
5,DoorDash,com.dd.doordash,ok,"DoorDash: Food, Grocery, More",4.657290,6029435,"50,000,000+",
6,Duolingo,com.duolingo,ok,Duolingo: Language Lessons,4.726992,47252796,"500,000,000+",
7,Google Maps,com.google.android.apps.maps,ok,Google Maps,3.248390,19469026,"10,000,000,000+",
8,Netflix,com.netflix.mediaclient,ok,Netflix,3.870866,15170397,"1,000,000,000+",
9,Reddit,com.reddit.frontpage,ok,Reddit,4.585813,4691886,"100,000,000+",


## 9. Helper functions for cleaning, keys, and quality flags

I keep the cleaning light on purpose.

For this phase, the main goal is ingestion reliability, duplicate handling, and operational tracking. This is not an NLP modeling notebook.

In [11]:
def normalize_text(value):
    if value is None:
        return None

    text = str(value)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def to_iso(value):
    if value is None:
        return None

    if isinstance(value, datetime):
        if value.tzinfo is None:
            return value.replace(tzinfo=timezone.utc).isoformat()
        return value.isoformat()

    return str(value)

def make_review_key(source, app_id, review_id, content, review_created_at, user_name):
    if review_id is not None and str(review_id).strip() != "":
        base = f"{source}|{app_id}|{review_id}"
    else:
        fallback = f"{source}|{app_id}|{content}|{review_created_at}|{user_name}"
        fallback_hash = hashlib.sha256(fallback.encode("utf-8")).hexdigest()
        base = f"{source}|{app_id}|missing_review_id|{fallback_hash}"

    return hashlib.sha256(base.encode("utf-8")).hexdigest()

def make_flag(review_key, run_id, app_id, flag_name, flag_severity, flag_value):
    flag_base = f"{review_key}|{run_id}|{flag_name}|{flag_value}"
    flag_id = hashlib.sha256(flag_base.encode("utf-8")).hexdigest()

    return {
        "flag_id": flag_id,
        "review_key": review_key,
        "run_id": run_id,
        "app_id": app_id,
        "flag_name": flag_name,
        "flag_severity": flag_severity,
        "flag_value": str(flag_value),
        "created_at": datetime.now(timezone.utc).isoformat()
    }

def get_quality_flags(review_row):
    flags = []

    review_key = review_row["review_key"]
    run_id = review_row["run_id"]
    app_id = review_row["app_id"]

    review_id = review_row.get("review_id")
    content_raw = review_row.get("content_raw")
    content_cleaned = normalize_text(content_raw)
    score = review_row.get("score")
    review_created_at = review_row.get("review_created_at")
    app_version = review_row.get("app_version")
    reply_content_raw = review_row.get("reply_content_raw")

    if review_id is None or str(review_id).strip() == "":
        flags.append(make_flag(review_key, run_id, app_id, "missing_review_id", "high", "missing"))

    if content_raw is None:
        flags.append(make_flag(review_key, run_id, app_id, "missing_content", "high", "missing"))
    elif content_cleaned == "":
        flags.append(make_flag(review_key, run_id, app_id, "empty_content", "medium", "empty"))

    if score is None:
        flags.append(make_flag(review_key, run_id, app_id, "missing_score", "high", "missing"))
    else:
        try:
            score_int = int(score)
            if score_int < 1 or score_int > 5:
                flags.append(make_flag(review_key, run_id, app_id, "invalid_score", "high", score_int))
        except Exception:
            flags.append(make_flag(review_key, run_id, app_id, "invalid_score", "high", score))

    if review_created_at is None or str(review_created_at).strip() == "":
        flags.append(make_flag(review_key, run_id, app_id, "missing_review_date", "high", "missing"))

    if app_version is None or str(app_version).strip() == "":
        flags.append(make_flag(review_key, run_id, app_id, "missing_app_version", "info", "missing"))

    if reply_content_raw is None or str(reply_content_raw).strip() == "":
        flags.append(make_flag(review_key, run_id, app_id, "missing_developer_reply", "info", "missing"))

    return flags

def raw_review_to_row(raw_review, app_name, app_id, fetched_at):
    review_id = raw_review.get("reviewId")
    content_raw = raw_review.get("content")
    review_created_at = to_iso(raw_review.get("at"))
    user_name = raw_review.get("userName")

    review_key = make_review_key(
        SOURCE,
        app_id,
        review_id,
        content_raw,
        review_created_at,
        user_name
    )

    app_version = raw_review.get("reviewCreatedVersion")

    if app_version is None:
        app_version = raw_review.get("appVersion")

    row = {
        "review_key": review_key,
        "source": SOURCE,
        "app_id": app_id,
        "app_name": app_name,
        "review_id": review_id,
        "user_name": user_name,
        "user_image": raw_review.get("userImage"),
        "content_raw": content_raw,
        "score": raw_review.get("score"),
        "thumbs_up_count": raw_review.get("thumbsUpCount"),
        "review_created_at": review_created_at,
        "reply_content_raw": raw_review.get("replyContent"),
        "replied_at": to_iso(raw_review.get("repliedAt")),
        "app_version": app_version,
        "fetched_at": fetched_at,
        "run_id": RUN_ID,
        "raw_json": json.dumps(raw_review, ensure_ascii=False, default=str)
    }

    return row

print("Review helper functions are ready.")

Review helper functions are ready.


## 10. Fetch reviews and insert new records

For each app, I fetch the newest reviews and insert only records that are not already in the database.

The duplicate rule is handled by the database key and by `INSERT OR IGNORE`.

So if the same review appears again in a later run, it will be skipped instead of inserted again.

In [12]:
app_summary_rows = []
all_fetched_rows_for_export = []

for app_name, app_id in APPS.items():
    print("\n" + "=" * 90)
    print(f"Starting app: {app_name} ({app_id})")
    print("=" * 90)

    app_start_time = time.perf_counter()
    fetched_at = datetime.now(timezone.utc).isoformat()

    records_fetched = 0
    unique_reviews_in_batch = 0
    duplicate_reviews_in_batch = 0
    new_records_inserted = 0
    duplicates_skipped = 0
    inserted_quality_flags_count = 0
    error_message = ""

    raw_review_rows = []
    batch_quality_flags = []

    try:
        result, continuation_token = reviews(
            app_id,
            lang=LANGUAGE,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=TARGET_REVIEWS_PER_APP
        )

        result = result or []
        records_fetched = len(result)

        for raw_review in result:
            row = raw_review_to_row(raw_review, app_name, app_id, fetched_at)
            raw_review_rows.append(row)
            all_fetched_rows_for_export.append(row)

        review_keys = [row["review_key"] for row in raw_review_rows]
        unique_reviews_in_batch = len(set(review_keys))
        duplicate_reviews_in_batch = records_fetched - unique_reviews_in_batch

        for row in raw_review_rows:
            flags_for_row = get_quality_flags(row)
            batch_quality_flags.extend(flags_for_row)

            cursor = conn.execute(
                """
                INSERT OR IGNORE INTO phase2_reviews_raw (
                    review_key,
                    source,
                    app_id,
                    app_name,
                    review_id,
                    user_name,
                    user_image,
                    content_raw,
                    score,
                    thumbs_up_count,
                    review_created_at,
                    reply_content_raw,
                    replied_at,
                    app_version,
                    fetched_at,
                    run_id,
                    raw_json
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
                """,
                (
                    row["review_key"],
                    row["source"],
                    row["app_id"],
                    row["app_name"],
                    row["review_id"],
                    row["user_name"],
                    row["user_image"],
                    row["content_raw"],
                    row["score"],
                    row["thumbs_up_count"],
                    row["review_created_at"],
                    row["reply_content_raw"],
                    row["replied_at"],
                    row["app_version"],
                    row["fetched_at"],
                    row["run_id"],
                    row["raw_json"]
                )
            )

            inserted_this_row = cursor.rowcount

            if inserted_this_row == 1:
                new_records_inserted += 1

                cleaned_text = normalize_text(row["content_raw"])
                content_length = len(cleaned_text) if cleaned_text is not None else 0

                if row["reply_content_raw"] is None or str(row["reply_content_raw"]).strip() == "":
                    has_developer_reply = 0
                else:
                    has_developer_reply = 1

                conn.execute(
                    """
                    INSERT OR REPLACE INTO phase2_reviews_cleaned (
                        review_key,
                        source,
                        app_id,
                        content_cleaned,
                        content_length,
                        has_developer_reply,
                        score,
                        review_created_at,
                        app_version,
                        cleaned_at,
                        run_id
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
                    """,
                    (
                        row["review_key"],
                        row["source"],
                        row["app_id"],
                        cleaned_text,
                        content_length,
                        has_developer_reply,
                        row["score"],
                        row["review_created_at"],
                        row["app_version"],
                        datetime.now(timezone.utc).isoformat(),
                        RUN_ID
                    )
                )

            # Quality flags are tracked for this run, even if the review already exists.
            # This gives a real run-level field quality summary.
            for flag in flags_for_row:
                flag_cursor = conn.execute(
                    """
                    INSERT OR IGNORE INTO phase2_quality_flags (
                        flag_id,
                        review_key,
                        run_id,
                        app_id,
                        flag_name,
                        flag_severity,
                        flag_value,
                        created_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?);
                    """,
                    (
                        flag["flag_id"],
                        flag["review_key"],
                        flag["run_id"],
                        flag["app_id"],
                        flag["flag_name"],
                        flag["flag_severity"],
                        flag["flag_value"],
                        flag["created_at"]
                    )
                )

                inserted_quality_flags_count += flag_cursor.rowcount

        duplicates_skipped = records_fetched - new_records_inserted

        review_dates = [
            row["review_created_at"]
            for row in raw_review_rows
            if row["review_created_at"] is not None
        ]

        if len(review_dates) > 0:
            min_review_date = min(review_dates)
            max_review_date = max(review_dates)
        else:
            min_review_date = None
            max_review_date = None

    except Exception as e:
        min_review_date = None
        max_review_date = None
        error_message = str(e)
        print("Error:", error_message)

    app_runtime_seconds = time.perf_counter() - app_start_time

    missing_review_id_count = sum(
        1 for flag in batch_quality_flags
        if flag["flag_name"] == "missing_review_id"
    )

    missing_content_count = sum(
        1 for flag in batch_quality_flags
        if flag["flag_name"] == "missing_content"
    )

    empty_content_count = sum(
        1 for flag in batch_quality_flags
        if flag["flag_name"] == "empty_content"
    )

    missing_score_count = sum(
        1 for flag in batch_quality_flags
        if flag["flag_name"] == "missing_score"
    )

    invalid_score_count = sum(
        1 for flag in batch_quality_flags
        if flag["flag_name"] == "invalid_score"
    )

    missing_review_date_count = sum(
        1 for flag in batch_quality_flags
        if flag["flag_name"] == "missing_review_date"
    )

    missing_app_version_count = sum(
        1 for flag in batch_quality_flags
        if flag["flag_name"] == "missing_app_version"
    )

    missing_developer_reply_count = sum(
        1 for flag in batch_quality_flags
        if flag["flag_name"] == "missing_developer_reply"
    )

    quality_flag_count = len(batch_quality_flags)

    summary_row = {
        "run_id": RUN_ID,
        "app_name": app_name,
        "app_id": app_id,
        "target_reviews": TARGET_REVIEWS_PER_APP,
        "records_fetched": records_fetched,
        "unique_reviews_in_batch": unique_reviews_in_batch,
        "duplicate_reviews_in_batch": duplicate_reviews_in_batch,
        "new_records_inserted": new_records_inserted,
        "duplicates_skipped": duplicates_skipped,
        "runtime_seconds": round(app_runtime_seconds, 2),
        "min_review_date": min_review_date,
        "max_review_date": max_review_date,
        "missing_review_id_count": missing_review_id_count,
        "missing_content_count": missing_content_count,
        "empty_content_count": empty_content_count,
        "missing_score_count": missing_score_count,
        "invalid_score_count": invalid_score_count,
        "missing_review_date_count": missing_review_date_count,
        "missing_app_version_count": missing_app_version_count,
        "missing_developer_reply_count": missing_developer_reply_count,
        "quality_flag_count": quality_flag_count,
        "quality_flags_inserted": inserted_quality_flags_count,
        "error_message": error_message
    }

    app_summary_rows.append(summary_row)

    conn.execute(
        """
        INSERT OR REPLACE INTO phase2_app_run_summary (
            run_id,
            app_name,
            app_id,
            target_reviews,
            records_fetched,
            unique_reviews_in_batch,
            duplicate_reviews_in_batch,
            new_records_inserted,
            duplicates_skipped,
            runtime_seconds,
            min_review_date,
            max_review_date,
            missing_review_id_count,
            missing_content_count,
            empty_content_count,
            missing_score_count,
            invalid_score_count,
            missing_review_date_count,
            missing_app_version_count,
            missing_developer_reply_count,
            quality_flag_count,
            error_message
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
        """,
        (
            summary_row["run_id"],
            summary_row["app_name"],
            summary_row["app_id"],
            summary_row["target_reviews"],
            summary_row["records_fetched"],
            summary_row["unique_reviews_in_batch"],
            summary_row["duplicate_reviews_in_batch"],
            summary_row["new_records_inserted"],
            summary_row["duplicates_skipped"],
            summary_row["runtime_seconds"],
            summary_row["min_review_date"],
            summary_row["max_review_date"],
            summary_row["missing_review_id_count"],
            summary_row["missing_content_count"],
            summary_row["empty_content_count"],
            summary_row["missing_score_count"],
            summary_row["invalid_score_count"],
            summary_row["missing_review_date_count"],
            summary_row["missing_app_version_count"],
            summary_row["missing_developer_reply_count"],
            summary_row["quality_flag_count"],
            summary_row["error_message"]
        )
    )

    conn.commit()

    print(f"Fetched: {records_fetched}")
    print(f"Inserted new: {new_records_inserted}")
    print(f"Duplicates skipped: {duplicates_skipped}")
    print(f"Quality flags in fetched batch: {quality_flag_count}")
    print(f"Quality flags inserted for this run: {inserted_quality_flags_count}")
    print(f"Runtime seconds: {app_runtime_seconds:.2f}")

    time.sleep(1)

app_summary_df = pd.DataFrame(app_summary_rows)

print("\nApp-level run summary:")
display(app_summary_df)


Starting app: YouTube (com.google.android.youtube)
Fetched: 1200
Inserted new: 1200
Duplicates skipped: 0
Quality flags in fetched batch: 1220
Quality flags inserted for this run: 1220
Runtime seconds: 1.27

Starting app: TikTok (com.zhiliaoapp.musically)
Fetched: 1200
Inserted new: 1200
Duplicates skipped: 0
Quality flags in fetched batch: 630
Quality flags inserted for this run: 630
Runtime seconds: 1.63

Starting app: Spotify (com.spotify.music)
Fetched: 1200
Inserted new: 1200
Duplicates skipped: 0
Quality flags in fetched batch: 1248
Quality flags inserted for this run: 1248
Runtime seconds: 1.34

Starting app: Instagram (com.instagram.android)
Fetched: 1200
Inserted new: 1200
Duplicates skipped: 0
Quality flags in fetched batch: 1577
Quality flags inserted for this run: 1577
Runtime seconds: 0.71

Starting app: Uber (com.ubercab)
Fetched: 1200
Inserted new: 1200
Duplicates skipped: 0
Quality flags in fetched batch: 1368
Quality flags inserted for this run: 1368
Runtime seconds: 

,run_id,app_name,app_id,target_reviews,records_fetched,unique_reviews_in_batch,duplicate_reviews_in_batch,new_records_inserted,duplicates_skipped,runtime_seconds,min_review_date,max_review_date,missing_review_id_count,missing_content_count,empty_content_count,missing_score_count,invalid_score_count,missing_review_date_count,missing_app_version_count,missing_developer_reply_count,quality_flag_count,quality_flags_inserted,error_message
0,phase2_day1_controlled_scale_20260708_034445,YouTube,com.google.android.youtube,1200,1200,1200,0,1200,0,1.27,2026-07-06T09:09:19+00:00,2026-07-07T03:44:48+00:00,0,0,0,0,0,0,20,1200,1220,1220,
1,phase2_day1_controlled_scale_20260708_034445,TikTok,com.zhiliaoapp.musically,1200,1200,1200,0,1200,0,1.63,2026-07-05T03:11:35+00:00,2026-07-07T03:43:24+00:00,0,0,0,0,0,0,449,181,630,630,
2,phase2_day1_controlled_scale_20260708_034445,Spotify,com.spotify.music,1200,1200,1200,0,1200,0,1.34,2026-07-05T12:54:10+00:00,2026-07-07T03:44:11+00:00,0,0,0,0,0,0,191,1057,1248,1248,
3,phase2_day1_controlled_scale_20260708_034445,Instagram,com.instagram.android,1200,1200,1200,0,1200,0,0.71,2026-07-06T13:35:02+00:00,2026-07-07T03:44:30+00:00,0,0,0,0,0,0,377,1200,1577,1577,
4,phase2_day1_controlled_scale_20260708_034445,Uber,com.ubercab,1200,1200,1200,0,1200,0,0.65,2026-07-04T03:52:16+00:00,2026-07-07T03:36:18+00:00,0,0,0,0,0,0,171,1197,1368,1368,
5,phase2_day1_controlled_scale_20260708_034445,DoorDash,com.dd.doordash,1200,1200,1200,0,1200,0,0.83,2026-06-28T23:28:16+00:00,2026-07-07T03:41:39+00:00,0,0,0,0,0,0,126,1200,1326,1326,
6,phase2_day1_controlled_scale_20260708_034445,Duolingo,com.duolingo,1200,1200,1200,0,1200,0,1.37,2026-07-06T04:53:03+00:00,2026-07-07T03:43:18+00:00,0,0,0,0,0,0,77,1200,1277,1277,
7,phase2_day1_controlled_scale_20260708_034445,Google Maps,com.google.android.apps.maps,1200,1200,1200,0,1200,0,0.85,2026-06-30T02:30:54+00:00,2026-07-07T03:30:28+00:00,0,0,0,0,0,0,30,935,965,965,
8,phase2_day1_controlled_scale_20260708_034445,Netflix,com.netflix.mediaclient,1200,1200,1200,0,1200,0,0.97,2026-06-27T05:44:07+00:00,2026-07-07T03:34:36+00:00,0,0,0,0,0,0,379,1200,1579,1579,
9,phase2_day1_controlled_scale_20260708_034445,Reddit,com.reddit.frontpage,1200,1200,1200,0,1200,0,0.81,2026-06-27T04:42:08+00:00,2026-07-07T03:27:55+00:00,0,0,0,0,0,0,243,1200,1443,1443,


## 11. Save fetched review export

The database is the main storage location.

I also save a CSV export for this run so the output is easy to inspect from GitHub.

In [13]:
fetched_export_df = pd.DataFrame(all_fetched_rows_for_export)

fetched_export_path = OUTPUT_DIR / f"{RUN_LABEL}_fetched_reviews_{run_timestamp}.csv"

if len(fetched_export_df) > 0:
    export_cols = [
        "run_id",
        "source",
        "app_name",
        "app_id",
        "review_id",
        "score",
        "review_created_at",
        "app_version",
        "content_raw",
        "reply_content_raw",
        "fetched_at"
    ]

    export_cols = [
        col for col in export_cols
        if col in fetched_export_df.columns
    ]

    fetched_export_df[export_cols].to_csv(fetched_export_path, index=False)
    print("Fetched review export saved to:", fetched_export_path)
else:
    print("No fetched reviews to export.")

Fetched review export saved to: outputs/phase2_day1_controlled_scale_fetched_reviews_20260708_034445.csv


## 12. Build the final run summary

This is the main operational summary John asked for.

It shows:

- run time
- apps included
- records fetched
- new records inserted
- duplicates skipped
- errors
- quality flag counts
- database growth

In [14]:
run_finished_at = datetime.now(timezone.utc)
run_finished_at_text = run_finished_at.isoformat()
runtime_seconds = (run_finished_at - run_started_at).total_seconds()

if len(app_summary_df) > 0:
    records_fetched_total = int(app_summary_df["records_fetched"].sum())
    new_records_inserted_total = int(app_summary_df["new_records_inserted"].sum())
    duplicates_skipped_total = int(app_summary_df["duplicates_skipped"].sum())
    errors_total = int((app_summary_df["error_message"].fillna("") != "").sum())
    quality_flag_total = int(app_summary_df["quality_flag_count"].sum())
    quality_flags_inserted = int(app_summary_df["quality_flags_inserted"].sum())
else:
    records_fetched_total = 0
    new_records_inserted_total = 0
    duplicates_skipped_total = 0
    errors_total = 0
    quality_flag_total = 0
    quality_flags_inserted = 0

apps_failed = ", ".join(
    app_summary_df.loc[
        app_summary_df["error_message"].fillna("") != "",
        "app_name"
    ].tolist()
)

db_size_after_mb = get_db_size_mb(DB_PATH)
db_size_growth_mb = db_size_after_mb - db_size_before_mb

review_rows_after = count_rows(conn, "phase2_reviews_raw")
review_rows_growth = review_rows_after - review_rows_before

if errors_total > 0:
    run_status = "completed_with_errors"
else:
    run_status = "completed"

conn.execute(
    """
    UPDATE phase2_ingestion_runs
    SET
        run_finished_at = ?,
        runtime_seconds = ?,
        status = ?,
        records_fetched_total = ?,
        new_records_inserted_total = ?,
        duplicates_skipped_total = ?,
        errors_total = ?,
        apps_failed = ?,
        quality_flag_total = ?,
        quality_flags_inserted = ?,
        db_size_after_mb = ?,
        db_size_growth_mb = ?,
        review_rows_after = ?,
        review_rows_growth = ?
    WHERE run_id = ?;
    """,
    (
        run_finished_at_text,
        runtime_seconds,
        run_status,
        records_fetched_total,
        new_records_inserted_total,
        duplicates_skipped_total,
        errors_total,
        apps_failed,
        quality_flag_total,
        quality_flags_inserted,
        db_size_after_mb,
        db_size_growth_mb,
        review_rows_after,
        review_rows_growth,
        RUN_ID
    )
)

conn.commit()

run_summary_df = pd.DataFrame([{
    "run_id": RUN_ID,
    "run_label": RUN_LABEL,
    "phase": PHASE,
    "frequency_label": FREQUENCY_LABEL,
    "source": SOURCE,
    "language": LANGUAGE,
    "country": COUNTRY,
    "target_reviews_per_app": TARGET_REVIEWS_PER_APP,
    "app_count": len(APPS),
    "apps_included": ", ".join(APPS.keys()),
    "run_started_at": run_started_at_text,
    "run_finished_at": run_finished_at_text,
    "runtime_seconds": round(runtime_seconds, 2),
    "status": run_status,
    "records_fetched_total": records_fetched_total,
    "new_records_inserted_total": new_records_inserted_total,
    "duplicates_skipped_total": duplicates_skipped_total,
    "errors_total": errors_total,
    "apps_failed": apps_failed,
    "quality_flag_total": quality_flag_total,
    "quality_flags_inserted": quality_flags_inserted,
    "db_size_before_mb": round(db_size_before_mb, 4),
    "db_size_after_mb": round(db_size_after_mb, 4),
    "db_size_growth_mb": round(db_size_growth_mb, 4),
    "review_rows_before": review_rows_before,
    "review_rows_after": review_rows_after,
    "review_rows_growth": review_rows_growth
}])

print("Run summary:")
display(run_summary_df)

Run summary:


,run_id,run_label,phase,frequency_label,source,language,country,target_reviews_per_app,app_count,apps_included,run_started_at,run_finished_at,runtime_seconds,status,records_fetched_total,new_records_inserted_total,duplicates_skipped_total,errors_total,apps_failed,quality_flag_total,quality_flags_inserted,db_size_before_mb,db_size_after_mb,db_size_growth_mb,review_rows_before,review_rows_after,review_rows_growth
0,phase2_day1_controlled_scale_20260708_034445,phase2_day1_controlled_scale,phase2,once_daily_baseline,google_play,en,us,1200,10,"YouTube, TikTok, Spotify, Instagram, Uber, DoorDash, Duolingo, Google Maps, Netflix, Reddit",2026-07-08T03:44:46.106977+00:00,2026-07-08T03:45:09.378094+00:00,23.27,completed,12000,12000,0,0,,12633,12633,1.832,25.4922,23.6602,0,12000,12000


## 13. Save run summary files

These CSV files make the run easy to review without opening the SQLite database manually.

In [15]:
run_summary_path = RUN_SUMMARY_DIR / f"phase2_day1_run_summary_{run_timestamp}.csv"
app_summary_path = RUN_SUMMARY_DIR / f"phase2_day1_app_level_summary_{run_timestamp}.csv"
history_path = RUN_SUMMARY_DIR / "phase2_run_summary_history.csv"

run_summary_df.to_csv(run_summary_path, index=False)
app_summary_df.to_csv(app_summary_path, index=False)

if history_path.exists():
    history_df = pd.read_csv(history_path)
    history_df = pd.concat([history_df, run_summary_df], ignore_index=True)
    history_df = history_df.drop_duplicates(subset=["run_id"], keep="last")
else:
    history_df = run_summary_df.copy()

history_df.to_csv(history_path, index=False)

print("Run summary saved to:", run_summary_path)
print("App-level summary saved to:", app_summary_path)
print("Run summary history saved to:", history_path)

Run summary saved to: outputs/run_summaries/phase2_day1_run_summary_20260708_034445.csv
App-level summary saved to: outputs/run_summaries/phase2_day1_app_level_summary_20260708_034445.csv
Run summary history saved to: outputs/run_summaries/phase2_run_summary_history.csv


## 14. Quality flag summary

This section summarizes field-level issues from the run.

Some flags are high severity, like missing review ID or missing score. Some are only informational, like missing developer reply.

In [16]:
quality_flag_summary_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        flag_name,
        flag_severity,
        COUNT(*) AS flag_count
    FROM phase2_quality_flags
    WHERE run_id = ?
    GROUP BY run_id, flag_name, flag_severity
    ORDER BY flag_severity, flag_name;
    """,
    conn,
    params=[RUN_ID]
)

quality_flag_summary_path = QUALITY_DIR / f"phase2_day1_quality_flag_summary_{run_timestamp}.csv"
quality_flag_summary_df.to_csv(quality_flag_summary_path, index=False)

print("Quality flag summary:")
display(quality_flag_summary_df)

print("Quality flag summary saved to:", quality_flag_summary_path)

Quality flag summary:


,run_id,flag_name,flag_severity,flag_count
0,phase2_day1_controlled_scale_20260708_034445,missing_app_version,info,2063
1,phase2_day1_controlled_scale_20260708_034445,missing_developer_reply,info,10570


Quality flag summary saved to: outputs/quality/phase2_day1_quality_flag_summary_20260708_034445.csv


## 15. Database relationship checks

This is a basic integrity check after the run.

The checks below look for:

- raw reviews without a matching app row
- cleaned reviews without a raw review row
- quality flags without a raw review row
- duplicate `source + app_id + review_id` combinations
- raw reviews from this run that are not linked to the ingestion run

In [17]:
raw_without_app = pd.read_sql_query(
    """
    SELECT COUNT(*) AS issue_count
    FROM phase2_reviews_raw r
    LEFT JOIN phase2_apps a
        ON r.app_id = a.app_id
    WHERE a.app_id IS NULL;
    """,
    conn
)["issue_count"].iloc[0]

cleaned_without_raw = pd.read_sql_query(
    """
    SELECT COUNT(*) AS issue_count
    FROM phase2_reviews_cleaned c
    LEFT JOIN phase2_reviews_raw r
        ON c.review_key = r.review_key
    WHERE r.review_key IS NULL;
    """,
    conn
)["issue_count"].iloc[0]

flags_without_raw = pd.read_sql_query(
    """
    SELECT COUNT(*) AS issue_count
    FROM phase2_quality_flags q
    LEFT JOIN phase2_reviews_raw r
        ON q.review_key = r.review_key
    WHERE r.review_key IS NULL;
    """,
    conn
)["issue_count"].iloc[0]

flags_without_run = pd.read_sql_query(
    """
    SELECT COUNT(*) AS issue_count
    FROM phase2_quality_flags q
    LEFT JOIN phase2_ingestion_runs ir
        ON q.run_id = ir.run_id
    WHERE q.run_id = ?
      AND ir.run_id IS NULL;
    """,
    conn,
    params=[RUN_ID]
)["issue_count"].iloc[0]

duplicate_review_keys = pd.read_sql_query(
    """
    SELECT COUNT(*) AS issue_count
    FROM (
        SELECT
            source,
            app_id,
            review_id,
            COUNT(*) AS cnt
        FROM phase2_reviews_raw
        WHERE review_id IS NOT NULL
          AND TRIM(review_id) != ''
        GROUP BY source, app_id, review_id
        HAVING COUNT(*) > 1
    );
    """,
    conn
)["issue_count"].iloc[0]

run_rows_without_run = pd.read_sql_query(
    """
    SELECT COUNT(*) AS issue_count
    FROM phase2_reviews_raw r
    LEFT JOIN phase2_ingestion_runs ir
        ON r.run_id = ir.run_id
    WHERE r.run_id = ?
      AND ir.run_id IS NULL;
    """,
    conn,
    params=[RUN_ID]
)["issue_count"].iloc[0]

relationship_checks = [
    {
        "check_name": "raw_reviews_have_app_record",
        "issue_count": int(raw_without_app),
        "status": "pass" if raw_without_app == 0 else "review_needed"
    },
    {
        "check_name": "cleaned_reviews_link_to_raw_reviews",
        "issue_count": int(cleaned_without_raw),
        "status": "pass" if cleaned_without_raw == 0 else "review_needed"
    },
    {
        "check_name": "quality_flags_link_to_raw_reviews",
        "issue_count": int(flags_without_raw),
        "status": "pass" if flags_without_raw == 0 else "review_needed"
    },
    {
        "check_name": "quality_flags_link_to_ingestion_run",
        "issue_count": int(flags_without_run),
        "status": "pass" if flags_without_run == 0 else "review_needed"
    },
    {
        "check_name": "no_duplicate_source_app_review_id",
        "issue_count": int(duplicate_review_keys),
        "status": "pass" if duplicate_review_keys == 0 else "review_needed"
    },
    {
        "check_name": "run_reviews_link_to_ingestion_run",
        "issue_count": int(run_rows_without_run),
        "status": "pass" if run_rows_without_run == 0 else "review_needed"
    }
]

relationship_checks_df = pd.DataFrame(relationship_checks)

relationship_check_path = RUN_SUMMARY_DIR / f"phase2_day1_relationship_checks_{run_timestamp}.csv"
relationship_checks_df.to_csv(relationship_check_path, index=False)

print("Database relationship checks:")
display(relationship_checks_df)

print("Relationship checks saved to:", relationship_check_path)

Database relationship checks:


,check_name,issue_count,status
0,raw_reviews_have_app_record,0,pass
1,cleaned_reviews_link_to_raw_reviews,0,pass
2,quality_flags_link_to_raw_reviews,0,pass
3,quality_flags_link_to_ingestion_run,0,pass
4,no_duplicate_source_app_review_id,0,pass
5,run_reviews_link_to_ingestion_run,0,pass


Relationship checks saved to: outputs/run_summaries/phase2_day1_relationship_checks_20260708_034445.csv


## 16. Compare database row counts before and after the run

This gives a simple view of database growth.

The database should grow mainly through new records, not duplicate records.

In [18]:
tables_after_df = get_tables(conn)

row_counts_after = []

for table_name in tables_after_df["name"]:
    row_counts_after.append({
        "table_name": table_name,
        "row_count_after": count_rows(conn, table_name)
    })

row_counts_after_df = pd.DataFrame(row_counts_after)

row_count_comparison_df = row_counts_before_df.merge(
    row_counts_after_df,
    on="table_name",
    how="outer"
).fillna(0)

row_count_comparison_df["row_count_before"] = row_count_comparison_df["row_count_before"].astype(int)
row_count_comparison_df["row_count_after"] = row_count_comparison_df["row_count_after"].astype(int)

row_count_comparison_df["row_growth"] = (
    row_count_comparison_df["row_count_after"] - row_count_comparison_df["row_count_before"]
)

row_count_comparison_path = RUN_SUMMARY_DIR / f"phase2_day1_database_row_growth_{run_timestamp}.csv"
row_count_comparison_df.to_csv(row_count_comparison_path, index=False)

print("Database row count comparison:")
display(row_count_comparison_df)

print("Database row growth file saved to:", row_count_comparison_path)

Database row count comparison:


,table_name,row_count_before,row_count_after,row_growth
0,app_sources,3,3,0
1,ingestion_run_targets,12,12,0
2,ingestion_runs,4,4,0
3,phase2_app_run_summary,10,20,10
4,phase2_apps,10,10,0
5,phase2_ingestion_runs,1,2,1
6,phase2_quality_flags,0,12633,12633
7,phase2_reviews_cleaned,0,12000,12000
8,phase2_reviews_raw,0,12000,12000
9,review_quality_flags,1200,1200,0


Database row growth file saved to: outputs/run_summaries/phase2_day1_database_row_growth_20260708_034445.csv


## 17. Write the Phase 2 Day 1 findings report

This report uses the actual outputs from the run.

I keep the language short because the CSV files and database have the detailed numbers.

In [19]:
def df_to_markdown_table(df):
    if df is None or len(df) == 0:
        return "_No rows._"

    safe_df = df.copy()
    safe_df = safe_df.fillna("")

    headers = list(safe_df.columns)
    lines = []

    lines.append("| " + " | ".join(headers) + " |")
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")

    for _, row in safe_df.iterrows():
        values = [
            str(row[col]).replace("\n", " ").replace("|", "/")
            for col in headers
        ]
        lines.append("| " + " | ".join(values) + " |")

    return "\n".join(lines)

short_app_summary_cols = [
    "app_name",
    "records_fetched",
    "new_records_inserted",
    "duplicates_skipped",
    "runtime_seconds",
    "quality_flag_count",
    "error_message"
]

short_app_summary_cols = [
    col for col in short_app_summary_cols
    if col in app_summary_df.columns
]

report_text = f"""
# Google Play Controlled Scale Ingestion — Phase 2 Day 1 Findings

## 1. Purpose

This run starts Phase 2 of the Google Play ingestion pipeline. The goal is to test whether the existing SQLite database pipeline can move from a small controlled demo into a larger recurring ingestion setup.

## 2. Test Scope

- Source: {SOURCE}
- Language / country: {LANGUAGE} / {COUNTRY}
- Apps tested: {len(APPS)}
- Target reviews per app: {TARGET_REVIEWS_PER_APP}
- Database used: `{DB_PATH}`
- Run ID: `{RUN_ID}`
- Frequency label: `{FREQUENCY_LABEL}`
- Duplicate rule: `source + app_id + review_id`

## 3. Run Summary

{df_to_markdown_table(run_summary_df)}

## 4. App-Level Summary

{df_to_markdown_table(app_summary_df[short_app_summary_cols])}

## 5. Quality Flag Summary

{df_to_markdown_table(quality_flag_summary_df)}

## 6. Database Relationship Checks

{df_to_markdown_table(relationship_checks_df)}

## 7. Main Notes

- The run continued from the existing SQLite database instead of starting from an empty file.
- New reviews were inserted through the Phase 2 raw and cleaned review tables.
- Duplicates were skipped using the database-level unique rule.
- Quality flags were linked back to raw reviews and the ingestion run.
- The run summary CSV shows runtime, apps included, records fetched, inserted rows, duplicates skipped, errors, quality flags, and database growth.

## 8. Next Step

The next controlled runs should use the same database and same app list, but run at different times. This will make it possible to compare once-daily and twice-daily collection behavior.
"""

report_path = REPORT_DIR / f"phase2_day1_controlled_scale_findings_{run_timestamp}.md"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text.strip())

print("Findings report saved to:", report_path)

Findings report saved to: reports/phase2_day1_controlled_scale_findings_20260708_034445.md


## 18. Files created or updated in this run

These are the main files to keep in GitHub after the notebook is finished.

In [20]:
created_files = [
    str(backup_path),
    str(run_summary_path),
    str(app_summary_path),
    str(history_path),
    str(quality_flag_summary_path),
    str(relationship_check_path),
    str(row_count_comparison_path),
    str(report_path),
    DB_PATH
]

if "fetched_export_path" in globals() and len(fetched_export_df) > 0:
    created_files.insert(1, str(fetched_export_path))

created_files_df = pd.DataFrame({
    "created_or_updated_file": created_files
})

display(created_files_df)

,created_or_updated_file
0,database/backups/google_play_reviews_before_phase2_day1_controlled_scale_20260708_034445.sqlite
1,outputs/phase2_day1_controlled_scale_fetched_reviews_20260708_034445.csv
2,outputs/run_summaries/phase2_day1_run_summary_20260708_034445.csv
3,outputs/run_summaries/phase2_day1_app_level_summary_20260708_034445.csv
4,outputs/run_summaries/phase2_run_summary_history.csv
5,outputs/quality/phase2_day1_quality_flag_summary_20260708_034445.csv
6,outputs/run_summaries/phase2_day1_relationship_checks_20260708_034445.csv
7,outputs/run_summaries/phase2_day1_database_row_growth_20260708_034445.csv
8,reports/phase2_day1_controlled_scale_findings_20260708_034445.md
9,database/google_play_reviews.sqlite


## 19. Quick GitHub check

Before committing, I check the changed files.

I do not commit automatically from the notebook because I want to review the outputs first.

In [21]:
!git status --short

 M database/google_play_reviews.sqlite
?? database/backups/
?? outputs/quality/
?? outputs/run_summaries/
?? reports/phase2_day1_controlled_scale_findings_20260708_034445.md


## 20. Next run plan

For the next Phase 2 runs, I should keep the same database and app list.

Recommended next runs:

- Phase 2 Day 1 evening: `FREQUENCY_LABEL = "twice_daily_pm"`
- Phase 2 Day 2 morning: `FREQUENCY_LABEL = "daily_followup"`
- Phase 2 Day 3 morning: `FREQUENCY_LABEL = "daily_followup"`
- optional Phase 2 Day 3 evening: `FREQUENCY_LABEL = "twice_daily_pm"`

This will make it possible to compare:

- same-day duplicate rate
- next-day new review capture
- once-daily vs twice-daily behavior
- runtime and database growth across repeated runs

In [22]:
conn.close()
print("Database connection closed. Phase 2 Day 1 notebook completed.")

Database connection closed. Phase 2 Day 1 notebook completed.


In [23]:
from pathlib import Path
import zipfile
from google.colab import files

# Make sure we are inside the repo folder
repo_dir = Path("/content/app-review-source-validation")

if not repo_dir.exists():
    raise FileNotFoundError("Repo folder not found. Please check if the repo was cloned correctly.")

# Files/folders needed for GitHub upload
files_to_include = []

# 1. Updated database
db_file = repo_dir / "database" / "google_play_reviews.sqlite"
if db_file.exists():
    files_to_include.append(db_file)
else:
    print("Missing database file:", db_file)

# 2. Run summaries
run_summary_dir = repo_dir / "outputs" / "run_summaries"
if run_summary_dir.exists():
    files_to_include.extend(run_summary_dir.glob("phase2_day1_*.csv"))
    files_to_include.extend(run_summary_dir.glob("phase2_run_summary_history.csv"))
else:
    print("Missing run summary folder:", run_summary_dir)

# 3. Quality summary
quality_dir = repo_dir / "outputs" / "quality"
if quality_dir.exists():
    files_to_include.extend(quality_dir.glob("phase2_day1_*.csv"))
else:
    print("Missing quality folder:", quality_dir)

# 4. Findings report
report_dir = repo_dir / "reports"
if report_dir.exists():
    files_to_include.extend(report_dir.glob("phase2_day1_controlled_scale_findings_*.md"))
else:
    print("Missing reports folder:", report_dir)

# Remove duplicates and sort
files_to_include = sorted(set(files_to_include))

print("Files found for download:")
for f in files_to_include:
    print("-", f.relative_to(repo_dir))

if len(files_to_include) == 0:
    raise FileNotFoundError("No Phase 2 Day 1 output files found. Please check whether the notebook ran successfully.")

# Create zip file
zip_path = Path("/content/phase2_day1_github_upload_files.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for f in files_to_include:
        arcname = f.relative_to(repo_dir)
        zipf.write(f, arcname)

print("\nZip file created:")
print(zip_path)

# Download zip to local computer
files.download(str(zip_path))

Files found for download:
- database/google_play_reviews.sqlite
- outputs/quality/phase2_day1_quality_flag_summary_20260708_034445.csv
- outputs/run_summaries/phase2_day1_app_level_summary_20260708_034445.csv
- outputs/run_summaries/phase2_day1_database_row_growth_20260708_034445.csv
- outputs/run_summaries/phase2_day1_relationship_checks_20260708_034445.csv
- outputs/run_summaries/phase2_day1_run_summary_20260708_034445.csv
- outputs/run_summaries/phase2_run_summary_history.csv
- reports/phase2_day1_controlled_scale_findings_20260707_035428.md
- reports/phase2_day1_controlled_scale_findings_20260708_034445.md

Zip file created:
/content/phase2_day1_github_upload_files.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>